# Feature Engineering v2: Aggregate First, Join Contact Last

This notebook rewrites the original feature engineering pipeline to fix issues found during EDA:

1. Avoid early data loss by aggregating activity first and joining contact metadata only after developer-level features are built.
2. Handle zero-inflated behavior with binary flags plus `LOG1P` features.
3. Replace nested cumulative windows with non-overlapping recency windows: `0_30d`, `30_90d`, and `90_180d`.
4. Add recency, velocity, rate, and interaction features.
5. Normalize persona lane scores and add entropy so mixed personas are represented more honestly.
6. Add feature validation checks after major table creation.

7. Add a capped, composite developer effort score and effort bands that are safer than raw `activity_score` alone.

Recommended final output: `dev_profile_final_v4`.


In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH)

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)

ACTIVITY_TABLE = "activity_final"
CONTACT_TABLE = "contact_final"

# Robust helper: use natural log of 1+x in DuckDB.
LOG1P = "LN(1 + {x})"

## 1. Confirm source tables and columns

In [2]:
tables = con.execute("SHOW TABLES").fetchdf()
display(tables)

available = set(tables.iloc[:, 0].astype(str))
missing = {ACTIVITY_TABLE, CONTACT_TABLE} - available
if missing:
    raise ValueError(f"Missing required table(s): {missing}")

def get_columns(table_name: str) -> set:
    return set(con.execute(f"DESCRIBE {table_name}").fetchdf()["column_name"].astype(str))

activity_cols = get_columns(ACTIVITY_TABLE)
contact_cols = get_columns(CONTACT_TABLE)

required_activity_cols = {"dev_contact", "activity_date", "activity"}
required_contact_cols = {"developer_id"}

if required_activity_cols - activity_cols:
    raise ValueError(f"Missing activity columns: {required_activity_cols - activity_cols}")
if required_contact_cols - contact_cols:
    raise ValueError(f"Missing contact columns: {required_contact_cols - contact_cols}")

for table in [ACTIVITY_TABLE, CONTACT_TABLE]:
    print(f"\n{table}")
    display(con.execute(f"SELECT COUNT(*) AS rows FROM {table}").fetchdf())
    display(con.execute(f"DESCRIBE {table}").fetchdf())

,name
0,activity_base
1,activity_clean
2,activity_enriched_v1
3,activity_final
4,activity_ontology_v1
5,activity_raw
6,activity_score_mapping_clean
7,activity_score_mapping_raw
8,activity_work
9,contact_clean



activity_final


,rows
0,69347501


,column_name,column_type,null,key,default,extra
0,dev_contact,VARCHAR,YES,None,None,None
1,activity,VARCHAR,YES,None,None,None
2,activity_name,VARCHAR,YES,None,None,None
3,activity_type,VARCHAR,YES,None,None,None
4,activity_role,VARCHAR,YES,None,None,None
5,activity_attendance,VARCHAR,YES,None,None,None
6,activity_score,DOUBLE,YES,None,None,None
7,activity_date,DATE,YES,None,None,None
8,activity_id,VARCHAR,YES,None,None,None
9,filepath,VARCHAR,YES,None,None,None



contact_final


,rows
0,8903197


,column_name,column_type,null,key,default,extra
0,developer_id,VARCHAR,YES,None,None,None
1,program_application_source,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,sub_region,VARCHAR,YES,None,None,None
5,zone,VARCHAR,YES,None,None,None
6,territory,VARCHAR,YES,None,None,None
7,organization_english_name,VARCHAR,YES,None,None,None
8,development_areas,VARCHAR,YES,None,None,None
9,other_development_areas,VARCHAR,YES,None,None,None


## 2. Source sanity checks

In [3]:
dup_name_col = "activity_name" if "activity_name" in activity_cols else "activity"
source_validation = con.execute(f"""
WITH activity_checks AS (
    SELECT
        COUNT(*) AS activity_rows,
        COUNT(DISTINCT CAST(dev_contact AS VARCHAR)) AS activity_developers,
        SUM(CASE WHEN dev_contact IS NULL OR TRIM(CAST(dev_contact AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_dev_contact,
        SUM(CASE WHEN activity_date IS NULL THEN 1 ELSE 0 END) AS missing_activity_date,
        SUM(CASE WHEN activity_date > CURRENT_DATE THEN 1 ELSE 0 END) AS future_activity_dates,
        COUNT(*) - COUNT(DISTINCT (
            COALESCE(CAST(dev_contact AS VARCHAR), '') || '|' ||
            COALESCE(CAST(activity_date AS VARCHAR), '') || '|' ||
            COALESCE(CAST(activity AS VARCHAR), '') || '|' ||
            COALESCE(CAST({dup_name_col} AS VARCHAR), '')
        )) AS possible_duplicate_activity_rows
    FROM {ACTIVITY_TABLE}
),
contact_checks AS (
    SELECT
        COUNT(*) AS contact_rows,
        COUNT(DISTINCT CAST(developer_id AS VARCHAR)) AS contact_developers,
        SUM(CASE WHEN developer_id IS NULL OR TRIM(CAST(developer_id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_developer_id,
        COUNT(*) - COUNT(DISTINCT CAST(developer_id AS VARCHAR)) AS duplicate_contact_rows
    FROM {CONTACT_TABLE}
)
SELECT * FROM activity_checks CROSS JOIN contact_checks
""").fetchdf()

display(source_validation)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,activity_rows,activity_developers,missing_dev_contact,missing_activity_date,future_activity_dates,possible_duplicate_activity_rows,contact_rows,contact_developers,missing_developer_id,duplicate_contact_rows
0,69347501,7660278,0.0,0.0,0.0,33695697,8903197,8903197,0.0,0


## 3. Build an activity-only base table

Important design change: this table does **not** join contact metadata. Activity features are created from activity rows first so unmatched contacts do not cause early population loss and duplicate contact rows do not multiply activity rows.

In [4]:
def activity_col_expr(col: str, out: str = None, cast: str = "VARCHAR", default: str = "NULL") -> str:
    out = out or col
    if col in activity_cols:
        return f"CAST(a.{col} AS {cast}) AS {out}"
    return f"CAST({default} AS {cast}) AS {out}"

activity_score_expr = (
    "LEAST(GREATEST(COALESCE(TRY_CAST(a.activity_score AS DOUBLE), 0.0), 0.0), 100.0) AS activity_score"
    if "activity_score" in activity_cols else
    "CAST(0.0 AS DOUBLE) AS activity_score"
)

con.execute(f"""
CREATE OR REPLACE TABLE activity_base_v2 AS
SELECT
    CAST(a.dev_contact AS VARCHAR) AS developer_id,
    CAST(a.activity_date AS DATE) AS activity_date,
    LOWER(TRIM(CAST(a.activity AS VARCHAR))) AS activity,
    {activity_col_expr('activity_name')},
    {activity_col_expr('activity_type')},
    {activity_col_expr('activity_role')},
    {activity_col_expr('activity_attendance')},
    {activity_score_expr},
    {activity_col_expr('filepath')},
    {activity_col_expr('lead_source')},
    {activity_col_expr('lead_source_details')},
    {activity_col_expr('nvidia_campaign_id')}
FROM {ACTIVITY_TABLE} a
WHERE a.dev_contact IS NOT NULL
  AND TRIM(CAST(a.dev_contact AS VARCHAR)) <> ''
  AND a.activity_date IS NOT NULL
""")

display(con.execute("""
SELECT
    (SELECT COUNT(*) FROM activity_final WHERE dev_contact IS NOT NULL AND TRIM(CAST(dev_contact AS VARCHAR)) <> '' AND activity_date IS NOT NULL) AS valid_source_activity_rows,
    COUNT(*) AS activity_base_rows,
    COUNT(DISTINCT developer_id) AS activity_base_developers,
    MIN(activity_date) AS min_activity_date,
    MAX(activity_date) AS max_activity_date,
    MIN(activity_score) AS min_activity_score,
    MAX(activity_score) AS max_activity_score
FROM activity_base_v2
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,valid_source_activity_rows,activity_base_rows,activity_base_developers,min_activity_date,max_activity_date,min_activity_score,max_activity_score
0,69347501,69347501,7660278,2020-01-01,2026-03-12,0.0,100.0


## 4. Deduplicate contact metadata separately

This table is used only for final enrichment and profile-text persona hints. It is not joined into activity rows.

In [5]:
def contact_select_expr(col: str, out: str = None, cast: str = "VARCHAR", default: str = "NULL") -> str:
    out = out or col
    if col in contact_cols:
        return f"CAST({col} AS {cast}) AS {out}"
    return f"CAST({default} AS {cast}) AS {out}"

order_terms = []
if "last_modified_date" in contact_cols:
    order_terms.append("last_modified_date DESC NULLS LAST")
if "created_date" in contact_cols:
    order_terms.append("created_date DESC NULLS LAST")
order_by = ", ".join(order_terms) if order_terms else "developer_id"

contact_fields = [
    "developer_id", "created_date", "first_activity_date", "last_activity_date",
    "development_areas", "fields_of_interest", "account_id", "account_type",
    "country", "region", "industry_segment_vertical", "program_application_source",
    "organization_english_name", "normalized_account_name", "wwfo_category", "wwfo_target_list"
]

select_list = []
for col in contact_fields:
    if col == "developer_id":
        select_list.append("CAST(developer_id AS VARCHAR) AS developer_id")
    elif col in {"created_date", "first_activity_date", "last_activity_date"} and col in contact_cols:
        select_list.append(f"CAST({col} AS DATE) AS {col}")
    else:
        select_list.append(contact_select_expr(col))

con.execute(f"""
CREATE OR REPLACE TABLE contact_one_row_v2 AS
WITH ranked AS (
    SELECT
        {', '.join(select_list)},
        ROW_NUMBER() OVER (PARTITION BY CAST(developer_id AS VARCHAR) ORDER BY {order_by}) AS rn
    FROM {CONTACT_TABLE}
    WHERE developer_id IS NOT NULL
      AND TRIM(CAST(developer_id AS VARCHAR)) <> ''
)
SELECT * EXCLUDE (rn)
FROM ranked
WHERE rn = 1
""")

display(con.execute("""
SELECT
    COUNT(*) AS contact_one_row_rows,
    COUNT(DISTINCT developer_id) AS contact_one_row_developers
FROM contact_one_row_v2
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contact_one_row_rows,contact_one_row_developers
0,8903197,8903197


## 5. Build deterministic activity dictionary, then label events

This replaces the previous event-level `activity_labeled_v2` logic. The dictionary is intentionally small: one row per `activity`, with exactly one `journey_signal`, one `effort_level`, and one base `modality` per activity.

Persona/topic context is still extracted from `activity_name`, `filepath`, and source details, but it is kept as a separate feature dimension and does **not** change the activity's journey classification.


Update: DevZone downloads now receive a contextual journey and effort override based on filepath so installers/toolkits are Build/High, docs are Discover/Passive, and other DevZone downloads are Evaluate/Moderate.

Important: this row-level effort label is not the final developer effort level. The final developer-level effort score is built later from capped volume, score, breadth, recency, and high-signal behavior so repeated low-value events do not dominate.



In [6]:
# Deterministic activity dictionary: one row per activity.
# This is the production-safe replacement for the old event-level ontology logic.
con.execute(r"""
CREATE OR REPLACE TABLE activity_dictionary_v2 AS
WITH activities AS (
    SELECT DISTINCT activity
    FROM activity_base_v2
    WHERE activity IS NOT NULL
)
SELECT
    activity,

    CASE
        -- Contribution / advocacy style signals.
        WHEN activity IN ('forum contributions') THEN 'Champion'
        WHEN activity IN ('user feedback', 'bugs filed') THEN 'Evaluate'

        -- Product / implementation behavior.
        WHEN activity IN ('hosted api', 'brev', 'ngc downloads', 'devzone downloads', 'sdk downloads') THEN 'Build'

        -- Formal learning.
        WHEN activity IN ('dli training') THEN 'Learn'
        WHEN activity IN ('webinars', 'on-demand views', 'conf. sessions live') THEN 'Learn'

        -- Events, membership, campaigns, and lightweight engagement.
        WHEN activity IN ('conference', 'other events', 'event registrations', 'program applications',
                          'dev program membership', 'product specific comms', 'contests') THEN 'Discover'

        -- Conservative fallback.
        ELSE 'Discover'
    END AS journey_signal,

    CASE
        WHEN activity IN ('forum contributions', 'hosted api', 'brev', 'ngc downloads', 'devzone downloads',
                          'sdk downloads', 'user feedback', 'bugs filed') THEN 'High'
        WHEN activity IN ('dli training', 'webinars', 'on-demand views', 'conf. sessions live',
                          'conference', 'other events', 'contests') THEN 'Moderate'
        ELSE 'Passive'
    END AS effort_level,

    CASE
        WHEN activity IN ('ngc downloads', 'devzone downloads', 'sdk downloads') THEN 'Download'
        WHEN activity IN ('hosted api') THEN 'Hosted API'
        WHEN activity IN ('brev') THEN 'Cloud Workspace'
        WHEN activity IN ('forum contributions') THEN 'Community'
        WHEN activity IN ('dli training') THEN 'Training'
        WHEN activity IN ('webinars', 'conference', 'other events', 'event registrations', 'conf. sessions live') THEN 'Event'
        ELSE 'Content'
    END AS modality
FROM activities
""")

# Join the dictionary to activity facts. This table is event-level by design, but labels are deterministic.
# Contextual persona/topic scores are kept separate from journey_signal and effort_level.
con.execute(r"""
CREATE OR REPLACE TABLE activity_labeled_v2 AS
WITH base AS (
    SELECT
        a.*,
        CASE
            WHEN a.activity = 'devzone downloads' THEN
                CASE
                    WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.exe'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%installer%'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%toolkit%'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%.deb'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%.rpm'
                    THEN 'Build'
                    WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.pdf'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%docs/%'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%documentation%'
                    THEN 'Discover'
                    ELSE 'Evaluate'
                END
            ELSE d.journey_signal
        END AS journey_signal,
        CASE
            WHEN a.activity = 'devzone downloads' THEN
                CASE
                    WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.exe'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%installer%'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%toolkit%'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%.deb'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%.rpm'
                    THEN 'High'
                    WHEN LOWER(COALESCE(a.filepath, '')) LIKE '%.pdf'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%docs/%'
                      OR LOWER(COALESCE(a.filepath, '')) LIKE '%documentation%'
                    THEN 'Passive'
                    ELSE 'Moderate'
                END
            ELSE d.effort_level
        END AS effort_level,
        d.modality,
        LOWER(
            COALESCE(a.activity_name, '') || ' ' ||
            COALESCE(a.filepath, '') || ' ' ||
            COALESCE(a.lead_source_details, '') || ' ' ||
            COALESCE(a.activity_type, '') || ' ' ||
            COALESCE(a.activity_role, '') || ' ' ||
            COALESCE(a.lead_source, '')
        ) AS persona_activity_text
    FROM activity_base_v2 a
    LEFT JOIN activity_dictionary_v2 d USING (activity)
),
scored AS (
    SELECT
        *,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'cuda|cudnn|rapids|nccl|cutlass|\bdali\b|gpu|accelerated|hpc') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS cuda_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'triton|tensorrt|nemo|\bnim\b|llm|large language|genai|generative|inference|model') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS genai_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'isaac|robot|ros|autonomous machine|jetson|edge ai|autonomous') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS robotics_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'omniverse|simulation|digital twin|modulus|render|rtx|graphics') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS simulation_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'dli|training|course|workshop|webinar|certification|learn|community') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS learning_community_activity_score
    FROM base
)
SELECT
    *,
    CASE
        WHEN GREATEST(cuda_activity_score, genai_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) = 0 THEN 'Other'
        WHEN cuda_activity_score >= GREATEST(genai_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'CUDA'
        WHEN genai_activity_score >= GREATEST(cuda_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'GenAI'
        WHEN robotics_activity_score >= GREATEST(cuda_activity_score, genai_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'Robotics'
        WHEN simulation_activity_score >= GREATEST(cuda_activity_score, genai_activity_score, robotics_activity_score, learning_community_activity_score) THEN 'Simulation'
        ELSE 'Learning_Community'
    END AS persona_hint,
    cuda_activity_score AS cuda_persona_score,
    genai_activity_score AS genai_persona_score,
    robotics_activity_score AS robotics_persona_score,
    simulation_activity_score AS simulation_persona_score,
    learning_community_activity_score AS learning_community_persona_score
FROM scored
""")

print('Dictionary size and coverage')
display(con.execute("""
SELECT
    COUNT(*) AS dictionary_rows,
    COUNT(DISTINCT activity) AS distinct_activities,
    SUM(CASE WHEN journey_signal IS NULL OR effort_level IS NULL OR modality IS NULL THEN 1 ELSE 0 END) AS missing_labels
FROM activity_dictionary_v2
""").fetchdf())

print('Dictionary determinism check: should return zero rows')
display(con.execute("""
SELECT
    activity,
    COUNT(DISTINCT journey_signal) AS n_journey_signals,
    COUNT(DISTINCT effort_level) AS n_effort_levels,
    COUNT(DISTINCT modality) AS n_modalities
FROM activity_dictionary_v2
GROUP BY activity
HAVING COUNT(DISTINCT journey_signal) > 1
    OR COUNT(DISTINCT effort_level) > 1
    OR COUNT(DISTINCT modality) > 1
ORDER BY activity
""").fetchdf())

print('DevZone contextual filepath override distribution')
display(con.execute("""
SELECT journey_signal, effort_level, COUNT(*) AS rows
FROM activity_labeled_v2
WHERE activity = 'devzone downloads'
GROUP BY 1,2
ORDER BY rows DESC
""").fetchdf())

print('Dictionary')
display(con.execute("""
SELECT *
FROM activity_dictionary_v2
ORDER BY journey_signal, effort_level, activity
""").fetchdf())

print('Event-level label distribution')
display(con.execute("""
SELECT journey_signal, effort_level, persona_hint, modality, COUNT(*) AS rows
FROM activity_labeled_v2
GROUP BY 1,2,3,4
ORDER BY rows DESC
LIMIT 30
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dictionary size and coverage


,dictionary_rows,distinct_activities,missing_labels
0,19,19,0.0


Dictionary determinism check: should return zero rows


,activity,n_journey_signals,n_effort_levels,n_modalities


...

,journey_signal,effort_level,rows
0,Build,High,29046368
1,Evaluate,Moderate,9225560
2,Discover,Passive,1470666


Dictionary


,activity,journey_signal,effort_level,modality
0,brev,Build,High,Cloud Workspace
1,devzone downloads,Build,High,Download
2,ngc downloads,Build,High,Download
3,forum contributions,Champion,High,Community
4,conference,Discover,Moderate,Event
5,contests,Discover,Moderate,Content
6,other events,Discover,Moderate,Event
7,dev program membership,Discover,Passive,Content
8,event registrations,Discover,Passive,Event
9,hackathons,Discover,Passive,Content


Event-level label distribution


,journey_signal,effort_level,persona_hint,modality,rows
0,Build,High,CUDA,Download,17653352
1,Discover,Passive,Other,Content,12244393
2,Build,High,Other,Download,8492207
3,Build,High,Robotics,Download,7871375
4,Evaluate,Moderate,CUDA,Download,4377225
5,Evaluate,Moderate,Other,Download,2460046
6,Build,High,GenAI,Download,2303688
7,Evaluate,Moderate,Robotics,Download,2208990
8,Discover,Passive,CUDA,Content,1374440
9,Discover,Passive,Robotics,Download,1154821


## 6. Developer universe and anchor date

In [7]:
con.execute(f"""
CREATE OR REPLACE TABLE developer_universe_v2 AS
SELECT DISTINCT developer_id FROM activity_base_v2 WHERE developer_id IS NOT NULL
UNION
SELECT DISTINCT developer_id FROM contact_one_row_v2 WHERE developer_id IS NOT NULL
""")

date_summary = con.execute("""
SELECT MIN(activity_date) AS min_activity_date, MAX(activity_date) AS max_activity_date
FROM activity_labeled_v2
""").fetchdf()
ANCHOR_DATE = date_summary.loc[0, "max_activity_date"]
print("ANCHOR_DATE:", ANCHOR_DATE)

display(con.execute("""
SELECT
    COUNT(*) AS universe_developers,
    SUM(CASE WHEN a.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS developers_with_activity,
    SUM(CASE WHEN c.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS developers_with_contact,
    SUM(CASE WHEN a.developer_id IS NOT NULL AND c.developer_id IS NULL THEN 1 ELSE 0 END) AS activity_without_contact,
    SUM(CASE WHEN a.developer_id IS NULL AND c.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS contact_without_activity
FROM developer_universe_v2 u
LEFT JOIN (SELECT DISTINCT developer_id FROM activity_base_v2) a USING (developer_id)
LEFT JOIN contact_one_row_v2 c USING (developer_id)
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ANCHOR_DATE: 2026-03-12 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,universe_developers,developers_with_activity,developers_with_contact,activity_without_contact,contact_without_activity
0,9379190,7660278.0,8903197.0,475993.0,1718912.0


## 7. Non-overlapping recency window features

The old notebook used cumulative windows. This version creates incremental windows to reduce collinearity:

- `0_30d`
- `30_90d`
- `90_180d`

In [8]:
WINDOWS = [
    ("0_30d", 0, 30),
    ("30_90d", 30, 90),
    ("90_180d", 90, 180),
]

def build_window_features(label: str, start_days_ago: int, end_days_ago: int) -> None:
    table_name = f"dev_features_{label}_v2"
    con.execute(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    WITH max_dt AS (SELECT MAX(activity_date) AS anchor_date FROM activity_labeled_v2),
    agg AS (
        SELECT
            developer_id,
            COUNT(*) AS activity_count,
            SUM(activity_score) AS activity_score_sum,
            AVG(activity_score) AS activity_score_avg,
            COUNT(DISTINCT activity_date) AS unique_activity_days,
            COUNT(DISTINCT activity) AS unique_activity_types,
            COUNT(DISTINCT modality) AS unique_modalities,
            COUNT(DISTINCT DATE_TRUNC('week', activity_date)) AS active_weeks,
            MIN(activity_date) AS first_activity_date_window,
            MAX(activity_date) AS last_activity_date_window,
            SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS discover_count,
            SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS learn_count,
            SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS evaluate_count,
            SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS build_count,
            SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS champion_count,
            SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS high_effort_count,
            SUM(CASE WHEN modality = 'Download' THEN 1 ELSE 0 END) AS download_count,
            SUM(CASE WHEN modality = 'Hosted API' THEN 1 ELSE 0 END) AS hosted_api_count,
            SUM(CASE WHEN modality = 'Cloud Workspace' THEN 1 ELSE 0 END) AS cloud_workspace_count,
            SUM(CASE WHEN modality = 'Community' THEN 1 ELSE 0 END) AS community_count,
            SUM(CASE WHEN modality = 'Training' THEN 1 ELSE 0 END) AS training_count,
            SUM(CASE WHEN modality = 'Event' THEN 1 ELSE 0 END) AS event_count,
            SUM(cuda_persona_score) AS cuda_score,
            SUM(genai_persona_score) AS genai_score,
            SUM(robotics_persona_score) AS robotics_score,
            SUM(simulation_persona_score) AS simulation_score,
            SUM(learning_community_persona_score) AS learning_community_score
        FROM activity_labeled_v2, max_dt
        WHERE activity_date > anchor_date - INTERVAL {end_days_ago} DAY
          AND activity_date <= anchor_date - INTERVAL {start_days_ago} DAY
        GROUP BY developer_id
    )
    SELECT
        u.developer_id,
        COALESCE(a.activity_count, 0) AS activity_count,
        COALESCE(a.activity_score_sum, 0) AS activity_score_sum,
        COALESCE(a.activity_score_avg, 0) AS activity_score_avg,
        COALESCE(a.unique_activity_days, 0) AS unique_activity_days,
        COALESCE(a.unique_activity_types, 0) AS unique_activity_types,
        COALESCE(a.unique_modalities, 0) AS unique_modalities,
        COALESCE(a.active_weeks, 0) AS active_weeks,
        a.first_activity_date_window,
        a.last_activity_date_window,
        CASE WHEN a.last_activity_date_window IS NULL THEN 1 ELSE 0 END AS is_missing_last_activity_window,
        DATE_DIFF('day', a.last_activity_date_window, (SELECT anchor_date FROM max_dt)) AS days_since_last_activity_window,
        COALESCE(a.discover_count, 0) AS discover_count,
        COALESCE(a.learn_count, 0) AS learn_count,
        COALESCE(a.evaluate_count, 0) AS evaluate_count,
        COALESCE(a.build_count, 0) AS build_count,
        COALESCE(a.champion_count, 0) AS champion_count,
        COALESCE(a.high_effort_count, 0) AS high_effort_count,
        COALESCE(a.download_count, 0) AS download_count,
        COALESCE(a.hosted_api_count, 0) AS hosted_api_count,
        COALESCE(a.cloud_workspace_count, 0) AS cloud_workspace_count,
        COALESCE(a.community_count, 0) AS community_count,
        COALESCE(a.training_count, 0) AS training_count,
        COALESCE(a.event_count, 0) AS event_count,
        COALESCE(a.cuda_score, 0) AS cuda_score,
        COALESCE(a.genai_score, 0) AS genai_score,
        COALESCE(a.robotics_score, 0) AS robotics_score,
        COALESCE(a.simulation_score, 0) AS simulation_score,
        COALESCE(a.learning_community_score, 0) AS learning_community_score,
        CASE WHEN COALESCE(a.activity_count, 0) > 0 THEN 1 ELSE 0 END AS has_activity,
        LN(1 + COALESCE(a.activity_count, 0)) AS log_activity_count,
        LN(1 + COALESCE(a.activity_score_sum, 0)) AS log_activity_score_sum,
        LN(1 + COALESCE(a.build_count, 0)) AS log_build_count,
        LN(1 + COALESCE(a.high_effort_count, 0)) AS log_high_effort_count,
        COALESCE(a.activity_count, 0) * 1.0 / NULLIF(COALESCE(a.unique_activity_days, 0), 0) AS activity_per_active_day,
        COALESCE(a.build_count, 0) * 1.0 / NULLIF(COALESCE(a.activity_count, 0), 0) AS build_share,
        COALESCE(a.high_effort_count, 0) * 1.0 / NULLIF(COALESCE(a.activity_count, 0), 0) AS high_effort_share
    FROM developer_universe_v2 u
    LEFT JOIN agg a USING (developer_id)
    """)

for label, start, end in WINDOWS:
    build_window_features(label, start, end)
    print(f"Built dev_features_{label}_v2")
    display(con.execute(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT developer_id) AS developers,
           AVG(has_activity) AS pct_with_activity,
           MAX(activity_count) AS max_activity_count
    FROM dev_features_{label}_v2
    """).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built dev_features_0_30d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9379190,9379190,0.044572,144228


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built dev_features_30_90d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9379190,9379190,0.045342,272221


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built dev_features_90_180d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9379190,9379190,0.055522,354569


## 8. Combine recency windows into one developer-level table

In [9]:
con.execute("""
CREATE OR REPLACE TABLE dev_recency_features_v2 AS
SELECT
    u.developer_id,

    f0.activity_count AS activity_count_0_30d,
    f1.activity_count AS activity_count_30_90d,
    f2.activity_count AS activity_count_90_180d,
    f0.has_activity AS has_activity_0_30d,
    f1.has_activity AS has_activity_30_90d,
    f2.has_activity AS has_activity_90_180d,
    f0.log_activity_count AS log_activity_count_0_30d,
    f1.log_activity_count AS log_activity_count_30_90d,
    f2.log_activity_count AS log_activity_count_90_180d,

    f0.build_count AS build_count_0_30d,
    f1.build_count AS build_count_30_90d,
    f2.build_count AS build_count_90_180d,
    f0.log_build_count AS log_build_count_0_30d,
    f1.log_build_count AS log_build_count_30_90d,
    f2.log_build_count AS log_build_count_90_180d,

    f0.high_effort_count AS high_effort_count_0_30d,
    f1.high_effort_count AS high_effort_count_30_90d,
    f2.high_effort_count AS high_effort_count_90_180d,

    f0.unique_activity_days AS unique_activity_days_0_30d,
    f0.unique_activity_types AS unique_activity_types_0_30d,
    f0.unique_modalities AS unique_modalities_0_30d,
    f0.activity_per_active_day AS activity_per_active_day_0_30d,
    f0.build_share AS build_share_0_30d,
    f0.high_effort_share AS high_effort_share_0_30d,

    f0.days_since_last_activity_window AS days_since_last_activity_0_30d,
    f0.is_missing_last_activity_window AS is_missing_last_activity_0_30d,

    -- Velocity and recency-decay features.
    f0.activity_count * 1.0 / NULLIF(f1.activity_count, 0) AS activity_velocity_0_30_vs_30_90,
    f0.build_count * 1.0 / NULLIF(f1.build_count, 0) AS build_velocity_0_30_vs_30_90,
    (0.60 * f0.activity_count + 0.30 * f1.activity_count + 0.10 * f2.activity_count) AS weighted_recent_activity,
    (0.60 * f0.build_count + 0.30 * f1.build_count + 0.10 * f2.build_count) AS weighted_recent_build,

    -- Interaction features.
    CASE WHEN f0.activity_count > 0 AND f0.build_count = 0 THEN 1 ELSE 0 END AS active_non_builder_0_30d,
    CASE WHEN f0.activity_count = 0 AND f1.activity_count > 0 THEN 1 ELSE 0 END AS newly_inactive_0_30d,
    CASE WHEN f0.build_count > 0 AND f0.activity_count <= 2 THEN 1 ELSE 0 END AS low_volume_builder_0_30d,
    CASE WHEN f0.high_effort_count > 0 THEN 1 ELSE 0 END AS has_high_effort_0_30d,
    CASE WHEN f0.build_count > 0 OR f0.hosted_api_count > 0 OR f0.cloud_workspace_count > 0 THEN 1 ELSE 0 END AS recent_build_flag,
    CASE WHEN f0.champion_count > 0 THEN 1 ELSE 0 END AS recent_champion_flag
FROM developer_universe_v2 u
LEFT JOIN dev_features_0_30d_v2 f0 USING (developer_id)
LEFT JOIN dev_features_30_90d_v2 f1 USING (developer_id)
LEFT JOIN dev_features_90_180d_v2 f2 USING (developer_id)
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    AVG(has_activity_0_30d) AS pct_active_0_30d,
    AVG(newly_inactive_0_30d) AS pct_newly_inactive_0_30d
FROM dev_recency_features_v2
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,developers,pct_active_0_30d,pct_newly_inactive_0_30d
0,9379190,9379190,0.044572,0.03801


## 9. Lifetime features with normalization, log transforms, and clipping helpers

Update: Lifetime features now include lifecycle classification columns from the colleague notebook: `user_type`, `max_stage_reached`, and `lifetime_devzone_download_count`.


In [10]:
con.execute("""
CREATE OR REPLACE TABLE dev_features_lifetime_v2 AS
WITH agg AS (
    SELECT
        developer_id,
        COUNT(*) AS lifetime_activity_count,
        SUM(activity_score) AS lifetime_activity_score_sum,
        AVG(activity_score) AS lifetime_activity_score_avg,
        COUNT(DISTINCT activity_date) AS lifetime_unique_activity_days,
        COUNT(DISTINCT activity) AS lifetime_unique_activity_types,
        COUNT(DISTINCT modality) AS lifetime_unique_modalities,
        COUNT(DISTINCT DATE_TRUNC('week', activity_date)) AS lifetime_active_weeks,
        MIN(activity_date) AS lifetime_first_activity_date,
        MAX(activity_date) AS lifetime_last_activity_date,
        SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS lifetime_discover_count,
        SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS lifetime_learn_count,
        SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS lifetime_evaluate_count,
        SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS lifetime_build_count,
        SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS lifetime_champion_count,
        SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS lifetime_high_effort_count,
        SUM(CASE WHEN activity = 'dli training' THEN 1 ELSE 0 END) AS lifetime_dli_training_count,
        SUM(CASE WHEN activity = 'webinars' THEN 1 ELSE 0 END) AS lifetime_webinar_count,
        SUM(CASE WHEN activity = 'forum contributions' THEN 1 ELSE 0 END) AS lifetime_forum_count,
        SUM(CASE WHEN activity = 'bugs filed' THEN 1 ELSE 0 END) AS lifetime_bug_count,
        SUM(CASE WHEN activity = 'hackathons' THEN 1 ELSE 0 END) AS lifetime_hackathon_count,
        SUM(CASE WHEN activity = 'model api' OR activity = 'hosted api' THEN 1 ELSE 0 END) AS lifetime_api_count,
        SUM(CASE WHEN activity = 'devzone downloads' THEN 1 ELSE 0 END) AS lifetime_devzone_download_count,
        SUM(CASE WHEN activity = 'ngc downloads' THEN 1 ELSE 0 END) AS lifetime_ngc_download_count,
        SUM(cuda_persona_score) AS cuda_score,
        SUM(genai_persona_score) AS genai_score,
        SUM(robotics_persona_score) AS robotics_score,
        SUM(simulation_persona_score) AS simulation_score,
        SUM(learning_community_persona_score) AS learning_community_score,
        SUM(CASE WHEN persona_hint = 'Other' THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END) AS other_persona_score
    FROM activity_labeled_v2
    GROUP BY developer_id
),
filled AS (
    SELECT
        u.developer_id,
        COALESCE(a.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(a.lifetime_activity_score_sum, 0) AS lifetime_activity_score_sum,
        COALESCE(a.lifetime_activity_score_avg, 0) AS lifetime_activity_score_avg,
        COALESCE(a.lifetime_unique_activity_days, 0) AS lifetime_unique_activity_days,
        COALESCE(a.lifetime_unique_activity_types, 0) AS lifetime_unique_activity_types,
        COALESCE(a.lifetime_unique_modalities, 0) AS lifetime_unique_modalities,
        COALESCE(a.lifetime_active_weeks, 0) AS lifetime_active_weeks,
        a.lifetime_first_activity_date,
        a.lifetime_last_activity_date,
        COALESCE(a.lifetime_discover_count, 0) AS lifetime_discover_count,
        COALESCE(a.lifetime_learn_count, 0) AS lifetime_learn_count,
        COALESCE(a.lifetime_evaluate_count, 0) AS lifetime_evaluate_count,
        COALESCE(a.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(a.lifetime_champion_count, 0) AS lifetime_champion_count,
        COALESCE(a.lifetime_high_effort_count, 0) AS lifetime_high_effort_count,
        COALESCE(a.lifetime_dli_training_count, 0) AS lifetime_dli_training_count,
        COALESCE(a.lifetime_webinar_count, 0) AS lifetime_webinar_count,
        COALESCE(a.lifetime_forum_count, 0) AS lifetime_forum_count,
        COALESCE(a.lifetime_bug_count, 0) AS lifetime_bug_count,
        COALESCE(a.lifetime_hackathon_count, 0) AS lifetime_hackathon_count,
        COALESCE(a.lifetime_api_count, 0) AS lifetime_api_count,
        COALESCE(a.lifetime_devzone_download_count, 0) AS lifetime_devzone_download_count,
        COALESCE(a.lifetime_ngc_download_count, 0) AS lifetime_ngc_download_count,
        COALESCE(a.cuda_score, 0) AS cuda_score,
        COALESCE(a.genai_score, 0) AS genai_score,
        COALESCE(a.robotics_score, 0) AS robotics_score,
        COALESCE(a.simulation_score, 0) AS simulation_score,
        COALESCE(a.learning_community_score, 0) AS learning_community_score,
        COALESCE(a.other_persona_score, 0) AS other_persona_score
    FROM developer_universe_v2 u
    LEFT JOIN agg a USING (developer_id)
),
p99 AS (
    SELECT
        APPROX_QUANTILE(lifetime_activity_count, 0.99) AS p99_activity_count,
        APPROX_QUANTILE(lifetime_build_count, 0.99) AS p99_build_count,
        APPROX_QUANTILE(lifetime_high_effort_count, 0.99) AS p99_high_effort_count,
        APPROX_QUANTILE(lifetime_activity_score_sum, 0.99) AS p99_activity_score_sum
    FROM filled
)
SELECT
    f.*,
    CASE
        WHEN f.lifetime_unique_activity_days = 1 THEN 'tourist'
        WHEN f.lifetime_build_count + f.lifetime_champion_count <= 2
          AND f.lifetime_high_effort_count = 0
          AND f.lifetime_devzone_download_count >= 1
        THEN 'free_email_user'
        ELSE 'real_user'
    END AS user_type,
    CASE
        WHEN f.lifetime_champion_count >= 1 THEN 'Champion'
        WHEN f.lifetime_build_count >= 1 THEN 'Build'
        WHEN f.lifetime_evaluate_count >= 1 THEN 'Evaluate'
        WHEN f.lifetime_learn_count >= 1 THEN 'Learn'
        WHEN f.lifetime_discover_count >= 1 THEN 'Discover'
        ELSE 'None'
    END AS max_stage_reached,
    CASE WHEN lifetime_activity_count > 0 THEN 1 ELSE 0 END AS has_lifetime_activity,
    LN(1 + lifetime_activity_count) AS log_lifetime_activity_count,
    LN(1 + lifetime_activity_score_sum) AS log_lifetime_activity_score_sum,
    LN(1 + lifetime_build_count) AS log_lifetime_build_count,
    LN(1 + lifetime_high_effort_count) AS log_lifetime_high_effort_count,
    LEAST(lifetime_activity_count, p99.p99_activity_count) AS clipped_lifetime_activity_count_p99,
    LEAST(lifetime_build_count, p99.p99_build_count) AS clipped_lifetime_build_count_p99,
    LEAST(lifetime_activity_score_sum, p99.p99_activity_score_sum) AS clipped_lifetime_activity_score_sum_p99,
    LN(1 + LEAST(lifetime_activity_count, p99.p99_activity_count)) AS log_clipped_lifetime_activity_count_p99,
    LN(1 + LEAST(lifetime_activity_score_sum, p99.p99_activity_score_sum)) AS log_clipped_lifetime_activity_score_sum_p99,
    lifetime_activity_count * 1.0 / NULLIF(lifetime_active_weeks, 0) AS activity_per_active_week_lifetime,
    lifetime_build_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS build_share_lifetime,
    lifetime_high_effort_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS high_effort_share_lifetime
FROM filled f
CROSS JOIN p99
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    MAX(lifetime_activity_count) AS max_raw_activity_count,
    MAX(clipped_lifetime_activity_count_p99) AS max_clipped_activity_count
FROM dev_features_lifetime_v2
""").fetchdf())

print('New lifecycle feature distributions')
display(con.execute("""
SELECT user_type, max_stage_reached, COUNT(*) AS developers
FROM dev_features_lifetime_v2
GROUP BY 1,2
ORDER BY developers DESC
LIMIT 30
""").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,developers,max_raw_activity_count,max_clipped_activity_count
0,9379190,9379190,2027375,115


New lifecycle feature distributions


,user_type,max_stage_reached,developers
0,tourist,Discover,1902773
1,real_user,None,1718912
2,real_user,Build,1495403
3,tourist,Build,1448466
4,tourist,Learn,895400
5,tourist,Evaluate,718180
6,real_user,Learn,547761
7,real_user,Discover,224967
8,free_email_user,Evaluate,191619
9,real_user,Evaluate,133628


## 10. Developer-level effort score and effort bands

This replaces raw `activity_score` as the main effort segmentation feature. It combines capped lifetime volume, capped lifetime score, activity breadth, distinct assets, recent activity, and high-signal activity types. The caps and `LN(1+x)` transforms keep extreme repeated downloads from dominating the segmentation.


In [11]:

con.execute("""
CREATE OR REPLACE TABLE dev_effort_level_v2 AS
WITH base AS (
    SELECT
        lf.developer_id,
        lf.lifetime_activity_count,
        lf.lifetime_activity_score_sum,
        lf.lifetime_unique_activity_types,
        lf.lifetime_unique_modalities,
        lf.lifetime_high_effort_count,
        lf.lifetime_build_count,
        lf.lifetime_champion_count,
        lf.lifetime_dli_training_count,
        lf.lifetime_webinar_count,
        lf.lifetime_forum_count,
        lf.lifetime_bug_count,
        lf.lifetime_hackathon_count,
        lf.lifetime_api_count,
        lf.lifetime_devzone_download_count,
        lf.lifetime_ngc_download_count,
        lf.log_clipped_lifetime_activity_count_p99,
        lf.log_clipped_lifetime_activity_score_sum_p99,
        COALESCE(r.activity_count_0_30d, 0) AS activity_count_0_30d,
        COALESCE(r.activity_count_30_90d, 0) AS activity_count_30_90d,
        COALESCE(r.activity_count_90_180d, 0) AS activity_count_90_180d,
        COALESCE(r.unique_activity_types_0_30d, 0) AS unique_activity_types_0_30d,
        COALESCE(r.recent_build_flag, 0) AS recent_build_flag,
        CASE
            WHEN COALESCE(r.activity_count_0_30d, 0) > 0 THEN 1.00
            WHEN COALESCE(r.activity_count_30_90d, 0) > 0 THEN 0.75
            WHEN COALESCE(r.activity_count_90_180d, 0) > 0 THEN 0.50
            ELSE 0.25
        END AS recency_weight
    FROM dev_features_lifetime_v2 lf
    LEFT JOIN dev_recency_features_v2 r USING (developer_id)
), scored AS (
    SELECT
        *,
        (
            0.30 * COALESCE(log_clipped_lifetime_activity_score_sum_p99, 0)
          + 0.20 * COALESCE(log_clipped_lifetime_activity_count_p99, 0)
          + 0.15 * LN(1 + COALESCE(lifetime_unique_activity_types, 0))
          + 0.10 * LN(1 + COALESCE(lifetime_unique_modalities, 0))
          + 0.15 * (
                CASE WHEN lifetime_dli_training_count > 0 THEN 1 ELSE 0 END
              + CASE WHEN lifetime_api_count > 0 THEN 1 ELSE 0 END
              + CASE WHEN lifetime_ngc_download_count > 0 THEN 1 ELSE 0 END
              + CASE WHEN lifetime_devzone_download_count > 0 THEN 1 ELSE 0 END
              + CASE WHEN lifetime_forum_count > 0 THEN 1 ELSE 0 END
              + CASE WHEN lifetime_bug_count > 0 THEN 1 ELSE 0 END
              + CASE WHEN lifetime_hackathon_count > 0 THEN 1 ELSE 0 END
            )
          + 0.10 * LN(1 + COALESCE(activity_count_0_30d, 0))
        ) * recency_weight AS developer_effort_score
    FROM base
), cutoffs AS (
    SELECT
        QUANTILE_CONT(developer_effort_score, 0.50) AS p50_effort,
        QUANTILE_CONT(developer_effort_score, 0.75) AS p75_effort,
        QUANTILE_CONT(developer_effort_score, 0.90) AS p90_effort
    FROM scored
    WHERE lifetime_activity_count > 0
)
SELECT
    s.*,
    CASE
        WHEN s.lifetime_activity_count = 0 THEN 'no activity'
        WHEN s.developer_effort_score >= c.p90_effort THEN 'very high effort'
        WHEN s.developer_effort_score >= c.p75_effort THEN 'high effort'
        WHEN s.developer_effort_score >= c.p50_effort THEN 'medium effort'
        ELSE 'low effort'
    END AS developer_effort_level,
    CASE
        WHEN s.lifetime_activity_count = 0 THEN 0
        WHEN s.developer_effort_score >= c.p90_effort THEN 4
        WHEN s.developer_effort_score >= c.p75_effort THEN 3
        WHEN s.developer_effort_score >= c.p50_effort THEN 2
        ELSE 1
    END AS developer_effort_rank
FROM scored s
CROSS JOIN cutoffs c
""")

print('Developer effort level distribution')
display(con.execute("""
SELECT
    developer_effort_level,
    developer_effort_rank,
    COUNT(*) AS developers,
    ROUND(AVG(developer_effort_score), 3) AS avg_effort_score,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_lifetime_activity_count,
    ROUND(AVG(lifetime_activity_score_sum), 2) AS avg_lifetime_activity_score_sum,
    ROUND(AVG(lifetime_unique_activity_types), 2) AS avg_lifetime_activity_types
FROM dev_effort_level_v2
GROUP BY 1,2
ORDER BY developer_effort_rank DESC
""").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Developer effort level distribution


,developer_effort_level,developer_effort_rank,developers,avg_effort_score,avg_lifetime_activity_count,avg_lifetime_activity_score_sum,avg_lifetime_activity_types
0,very high effort,4,766295,1.226,57.20,180.79,2.56
1,high effort,3,1192087,0.550,9.55,47.57,2.34
2,medium effort,2,1950701,0.389,3.69,14.85,2.29
3,low effort,1,3751195,0.233,1.85,3.81,1.53
4,no activity,0,1718912,0.000,0.00,0.00,0.00


## 10. Contact-profile persona hints, then final persona

Activity-based persona is still primary. Contact profile text is added only after developer-level aggregation so it enriches persona without changing activity row counts.

In [12]:
con.execute(r"""
CREATE OR REPLACE TABLE dev_contact_persona_v2 AS
WITH base AS (
    SELECT
        developer_id,
        LOWER(
            COALESCE(development_areas, '') || ' ' ||
            COALESCE(fields_of_interest, '') || ' ' ||
            COALESCE(industry_segment_vertical, '')
        ) AS profile_text
    FROM contact_one_row_v2
)
SELECT
    developer_id,
    CASE WHEN REGEXP_MATCHES(profile_text, 'cuda|gpu|accelerated|hpc|rapids|cudnn') THEN 1.0 ELSE 0.0 END AS cuda_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'genai|generative|llm|ai|machine learning|deep learning|inference') THEN 1.0 ELSE 0.0 END AS genai_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'robot|isaac|ros|jetson|autonomous|edge') THEN 1.0 ELSE 0.0 END AS robotics_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'simulation|omniverse|digital twin|graphics|render|rtx') THEN 1.0 ELSE 0.0 END AS simulation_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'training|education|student|academic|community|developer program') THEN 1.0 ELSE 0.0 END AS learning_community_profile_score
FROM base
""")

con.execute("""
CREATE OR REPLACE TABLE dev_persona_v2 AS
WITH base AS (
    SELECT
        lf.developer_id,
        lf.cuda_score + COALESCE(cp.cuda_profile_score, 0) AS cuda_score,
        lf.genai_score + COALESCE(cp.genai_profile_score, 0) AS genai_score,
        lf.robotics_score + COALESCE(cp.robotics_profile_score, 0) AS robotics_score,
        lf.simulation_score + COALESCE(cp.simulation_profile_score, 0) AS simulation_score,
        lf.learning_community_score + COALESCE(cp.learning_community_profile_score, 0) AS learning_community_score,
        lf.other_persona_score
    FROM dev_features_lifetime_v2 lf
    LEFT JOIN dev_contact_persona_v2 cp USING (developer_id)
),
norm AS (
    SELECT *,
        cuda_score + genai_score + robotics_score + simulation_score + learning_community_score AS specific_persona_score,
        cuda_score + genai_score + robotics_score + simulation_score + learning_community_score + other_persona_score AS total_persona_score
    FROM base
),
shares AS (
    SELECT *,
        COALESCE(cuda_score / NULLIF(specific_persona_score, 0), 0) AS cuda_share,
        COALESCE(genai_score / NULLIF(specific_persona_score, 0), 0) AS genai_share,
        COALESCE(robotics_score / NULLIF(specific_persona_score, 0), 0) AS robotics_share,
        COALESCE(simulation_score / NULLIF(specific_persona_score, 0), 0) AS simulation_share,
        COALESCE(learning_community_score / NULLIF(specific_persona_score, 0), 0) AS learning_community_share,
        COALESCE(other_persona_score / NULLIF(total_persona_score, 0), 0) AS other_share
    FROM norm
),
entropy AS (
    SELECT *,
        -1 * (
            CASE WHEN cuda_share > 0 THEN cuda_share * LN(cuda_share) ELSE 0 END +
            CASE WHEN genai_share > 0 THEN genai_share * LN(genai_share) ELSE 0 END +
            CASE WHEN robotics_share > 0 THEN robotics_share * LN(robotics_share) ELSE 0 END +
            CASE WHEN simulation_share > 0 THEN simulation_share * LN(simulation_share) ELSE 0 END +
            CASE WHEN learning_community_share > 0 THEN learning_community_share * LN(learning_community_share) ELSE 0 END
        ) / LN(5) AS persona_entropy
    FROM shares
),
long_scores AS (
    SELECT developer_id, 'CUDA' AS persona, cuda_share AS score FROM entropy
    UNION ALL SELECT developer_id, 'GenAI', genai_share FROM entropy
    UNION ALL SELECT developer_id, 'Robotics', robotics_share FROM entropy
    UNION ALL SELECT developer_id, 'Simulation', simulation_share FROM entropy
    UNION ALL SELECT developer_id, 'Learning_Community', learning_community_share FROM entropy
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY developer_id ORDER BY score DESC, persona) AS rn,
        LEAD(score) OVER (PARTITION BY developer_id ORDER BY score DESC, persona) AS second_score
    FROM long_scores
)
SELECT
    e.*,
    CASE WHEN e.specific_persona_score = 0 THEN 'Unknown' ELSE r.persona END AS persona,
    CASE WHEN e.specific_persona_score = 0 THEN 0 ELSE r.score END AS persona_confidence,
    CASE
        WHEN e.specific_persona_score = 0 THEN 'Unknown'
        WHEN r.score >= 0.70 THEN 'High'
        WHEN r.score >= 0.45 THEN 'Medium'
        ELSE 'Low'
    END AS persona_confidence_tier,
    CASE
        WHEN e.specific_persona_score = 0 THEN 0
        WHEN e.persona_entropy >= 0.60 OR r.score - COALESCE(r.second_score, 0) <= 0.15 THEN 1
        ELSE 0
    END AS mixed_persona_flag
FROM entropy e
LEFT JOIN ranked r ON e.developer_id = r.developer_id AND r.rn = 1
""")

display(con.execute("""
SELECT persona, persona_confidence_tier, mixed_persona_flag, COUNT(*) AS developers
FROM dev_persona_v2
GROUP BY 1,2,3
ORDER BY developers DESC
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,persona,persona_confidence_tier,mixed_persona_flag,developers
0,Unknown,Unknown,0,1918828
1,CUDA,High,0,1834363
2,GenAI,High,0,1377082
3,CUDA,Medium,1,718196
4,CUDA,Medium,0,662938
5,GenAI,Medium,1,584715
6,Simulation,High,0,511271
7,Robotics,High,0,368202
8,CUDA,Low,1,317655
9,Learning_Community,High,0,314111


## 11. Journey state features from the recent window

For predictive modeling, avoid using journey-state labels if the target is derived from the same future/current activity window. Use the underlying count/rate features instead, or build journey state from a strictly prior window.

Dormancy / activation status
    ↓
Recent activity
    ↓
Intent signal
    ↓
Depth of behavior
    ↓
Trend
    ↓
Journey stage

In [13]:
con.execute("""
CREATE OR REPLACE TABLE dev_journey_state_v2 AS
WITH base AS (
    SELECT
        r.developer_id,

        -- Recent activity
        COALESCE(r.activity_count_0_30d, 0) AS activity_count_0_30d,
        COALESCE(r.activity_count_30_90d, 0) AS activity_count_30_90d,
        COALESCE(r.activity_count_90_180d, 0) AS activity_count_90_180d,

        COALESCE(r.build_count_0_30d, 0) AS build_count_0_30d,
        COALESCE(r.build_count_30_90d, 0) AS build_count_30_90d,

        COALESCE(r.high_effort_count_0_30d, 0) AS high_effort_count_0_30d,
        COALESCE(r.high_effort_count_30_90d, 0) AS high_effort_count_30_90d,

        COALESCE(r.unique_activity_types_0_30d, 0) AS unique_activity_types_0_30d,
        COALESCE(r.unique_modalities_0_30d, 0) AS unique_modalities_0_30d,

        COALESCE(r.recent_build_flag, 0) AS recent_build_flag,
        COALESCE(r.recent_champion_flag, 0) AS recent_champion_flag,

        -- Dormancy / activation
        COALESCE(d.is_activated, 0) AS is_activated,
        COALESCE(d.dormancy_status, 'Unactivated') AS dormancy_status,
        COALESCE(d.dormant_flag, 0) AS dormant_flag,
        COALESCE(d.at_risk_flag, 0) AS at_risk_flag,

        -- Lifetime context
        COALESCE(l.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(l.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(l.lifetime_champion_count, 0) AS lifetime_champion_count,

        -- Trend
        CASE
            WHEN COALESCE(r.activity_count_30_90d, 0) = 0
             AND COALESCE(r.activity_count_0_30d, 0) > 0 THEN 2.0
            WHEN COALESCE(r.activity_count_30_90d, 0) = 0 THEN 0.0
            ELSE CAST(r.activity_count_0_30d AS DOUBLE)
                 / NULLIF(CAST(r.activity_count_30_90d AS DOUBLE), 0)
        END AS recent_activity_trend_ratio

    FROM dev_recency_features_v2 r
    LEFT JOIN dev_dormancy_status_v2 d
        ON r.developer_id = d.developer_id
    LEFT JOIN dev_features_lifetime_v2 l
        ON r.developer_id = l.developer_id
),

scored AS (
    SELECT
        *,

        CASE
            WHEN activity_count_0_30d >= 5 THEN 'High'
            WHEN activity_count_0_30d >= 2 THEN 'Medium'
            WHEN activity_count_0_30d = 1 THEN 'Low'
            ELSE 'None'
        END AS activity_volume_band,

        CASE
            WHEN build_count_0_30d > 0 OR recent_build_flag = 1 THEN 'Build_Intent'
            WHEN high_effort_count_0_30d > 0 THEN 'Evaluation_Intent'
            WHEN activity_count_0_30d > 0 THEN 'Learning_Intent'
            ELSE 'No_Recent_Intent'
        END AS intent_signal,

        CASE
            WHEN recent_activity_trend_ratio >= 1.5 THEN 'Accelerating'
            WHEN recent_activity_trend_ratio >= 0.7 THEN 'Stable'
            WHEN recent_activity_trend_ratio > 0 THEN 'Declining'
            ELSE 'Inactive'
        END AS trend_signal

    FROM base
)

SELECT
    developer_id,

    CASE
        -- Dormancy should override effort
        WHEN is_activated = 0 THEN 'Unactivated'
        WHEN dormancy_status = 'Dormant' THEN 'Dormant'
        WHEN dormancy_status = 'At_Risk' THEN 'At_Risk'

        -- Highest-intent users
        WHEN recent_champion_flag = 1
          OR lifetime_champion_count > 0
        THEN 'Champion'

        -- Active builders
        WHEN build_count_0_30d >= 2
          OR recent_build_flag = 1
        THEN 'Builder'

        -- High-intent evaluation, but not yet building
        WHEN high_effort_count_0_30d > 0
          AND build_count_0_30d = 0
        THEN 'Evaluator'

        -- Active but mostly low-depth usage
        WHEN activity_count_0_30d > 0
          AND unique_activity_types_0_30d >= 2
        THEN 'Explorer'

        -- Minimal recent activity
        WHEN activity_count_0_30d > 0
        THEN 'Learner'

        -- Activated historically, but no recent activity and not yet dormant
        ELSE 'Cooling'
    END AS current_journey_state_30d,

    CASE
        WHEN is_activated = 0 THEN 0
        WHEN dormancy_status = 'Dormant' THEN 1
        WHEN dormancy_status = 'At_Risk' THEN 2
        WHEN activity_count_0_30d = 0 THEN 3
        WHEN activity_count_0_30d > 0 AND build_count_0_30d = 0 THEN 4
        WHEN build_count_0_30d > 0 OR recent_build_flag = 1 THEN 5
        WHEN recent_champion_flag = 1 OR lifetime_champion_count > 0 THEN 6
        ELSE 3
    END AS current_journey_rank_30d,

    activity_volume_band,
    intent_signal,
    trend_signal,
    recent_activity_trend_ratio

FROM scored
""")

display(con.execute("""
SELECT
    current_journey_state_30d,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_journey_state_v2
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

CatalogException: Catalog Error: Table with name dev_dormancy_status_v2 does not exist!
Did you mean "dev_dormancy_status_v1"?

LINE 45:     LEFT JOIN dev_dormancy_status_v2 d
                       ^

## 12. Behavior journey stage and lifecycle overlay

This block separates behavior from lifecycle:

- `behavior_journey_stage_30d`: what the developer appears to do, based on behavior and intent.
- `dormancy_status`: lifecycle/recency state, created separately.
- `current_journey_state_30d`: optional combined view for reporting, not the primary modeling label.

This avoids making journey stage a simple relabeling of dormancy.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dev_journey_state_v2 AS
WITH base AS (
    SELECT
        r.developer_id,

        -- Recent activity
        COALESCE(r.activity_count_0_30d, 0) AS activity_count_0_30d,
        COALESCE(r.activity_count_30_90d, 0) AS activity_count_30_90d,
        COALESCE(r.activity_count_90_180d, 0) AS activity_count_90_180d,
        COALESCE(r.build_count_0_30d, 0) AS build_count_0_30d,
        COALESCE(r.build_count_30_90d, 0) AS build_count_30_90d,
        COALESCE(r.high_effort_count_0_30d, 0) AS high_effort_count_0_30d,
        COALESCE(r.high_effort_count_30_90d, 0) AS high_effort_count_30_90d,
        COALESCE(r.unique_activity_types_0_30d, 0) AS unique_activity_types_0_30d,
        COALESCE(r.unique_modalities_0_30d, 0) AS unique_modalities_0_30d,
        COALESCE(r.recent_build_flag, 0) AS recent_build_flag,
        COALESCE(r.recent_champion_flag, 0) AS recent_champion_flag,

        -- Lifecycle / dormancy
        COALESCE(d.is_activated, 0) AS is_activated,
        COALESCE(d.lifetime_activity_count_for_activation, 0) AS lifetime_activity_count_for_activation,
        COALESCE(d.dormancy_status, 'Unactivated') AS dormancy_status,
        COALESCE(d.dormant_flag, 0) AS dormant_flag,
        COALESCE(d.at_risk_flag, 0) AS at_risk_flag,
        COALESCE(d.cooling_flag, 0) AS cooling_flag,
        d.days_since_last_activity,

        -- Lifetime context
        COALESCE(l.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(l.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(l.lifetime_champion_count, 0) AS lifetime_champion_count,
        COALESCE(l.lifetime_high_effort_count, 0) AS lifetime_high_effort_count,
        COALESCE(l.lifetime_unique_activity_types, 0) AS lifetime_unique_activity_types,
        COALESCE(l.lifetime_unique_modalities, 0) AS lifetime_unique_modalities,

        -- Trend only meaningful when there is activity in either comparison window.
        CASE
            WHEN COALESCE(r.activity_count_0_30d, 0) = 0 AND COALESCE(r.activity_count_30_90d, 0) = 0 THEN NULL
            WHEN COALESCE(r.activity_count_30_90d, 0) = 0 AND COALESCE(r.activity_count_0_30d, 0) > 0 THEN 2.0
            ELSE CAST(r.activity_count_0_30d AS DOUBLE) / NULLIF(CAST(r.activity_count_30_90d AS DOUBLE), 0)
        END AS recent_activity_trend_ratio

    FROM dev_recency_features_v2 r
    LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
    LEFT JOIN dev_features_lifetime_v2 l USING (developer_id)
),
scored AS (
    SELECT
        *,
        CASE
            WHEN activity_count_0_30d >= 5 THEN 'High'
            WHEN activity_count_0_30d >= 2 THEN 'Medium'
            WHEN activity_count_0_30d = 1 THEN 'Low'
            ELSE 'None'
        END AS activity_volume_band,

        -- Intent uses recent behavior first, then lifetime fallback for inactive historical users.
        CASE
            WHEN build_count_0_30d > 0 OR recent_build_flag = 1 OR lifetime_build_count >= 5 THEN 'Build_Intent'
            WHEN high_effort_count_0_30d > 0 OR lifetime_high_effort_count >= 5 THEN 'Evaluation_Intent'
            WHEN activity_count_0_30d > 0 THEN 'Learning_Intent'
            ELSE 'No_Recent_Intent'
        END AS intent_signal,

        CASE
            WHEN recent_activity_trend_ratio IS NULL THEN NULL
            WHEN recent_activity_trend_ratio >= 1.5 THEN 'Accelerating'
            WHEN recent_activity_trend_ratio >= 0.7 THEN 'Stable'
            WHEN recent_activity_trend_ratio > 0 THEN 'Declining'
            ELSE NULL
        END AS trend_signal
    FROM base
),
behavior AS (
    SELECT
        *,
        CASE
            WHEN lifetime_activity_count = 0 THEN 'Unactivated'
            WHEN build_count_0_30d > 0 OR recent_build_flag = 1 OR lifetime_build_count >= 10 THEN 'Builder'
            WHEN high_effort_count_0_30d > 0 OR lifetime_high_effort_count >= 10 THEN 'Evaluator'
            WHEN activity_count_0_30d > 0 AND unique_activity_types_0_30d >= 2 THEN 'Explorer'
            WHEN activity_count_0_30d > 0 THEN 'Learner'
            WHEN lifetime_activity_count > 0 THEN 'Historically_Active'
            ELSE 'Unactivated'
        END AS behavior_journey_stage_30d
    FROM scored
)
SELECT
    developer_id,
    behavior_journey_stage_30d,

    -- Optional combined reporting view. Use behavior_journey_stage_30d + dormancy_status for modeling.
    CASE
        WHEN behavior_journey_stage_30d = 'Unactivated' THEN 'Unactivated'
        WHEN dormancy_status IN ('Dormant', 'At_Risk', 'Cooling') THEN dormancy_status || '_' || behavior_journey_stage_30d
        ELSE behavior_journey_stage_30d
    END AS current_journey_state_30d,

    CASE
        WHEN behavior_journey_stage_30d = 'Unactivated' THEN 0
        WHEN behavior_journey_stage_30d = 'Historically_Active' THEN 1
        WHEN behavior_journey_stage_30d = 'Learner' THEN 2
        WHEN behavior_journey_stage_30d = 'Explorer' THEN 3
        WHEN behavior_journey_stage_30d = 'Evaluator' THEN 4
        WHEN behavior_journey_stage_30d = 'Builder' THEN 5
        ELSE 1
    END AS behavior_journey_rank_30d,

    -- Keep existing name for compatibility with downstream notebooks.
    CASE
        WHEN behavior_journey_stage_30d = 'Unactivated' THEN 0
        WHEN behavior_journey_stage_30d = 'Historically_Active' THEN 1
        WHEN behavior_journey_stage_30d = 'Learner' THEN 2
        WHEN behavior_journey_stage_30d = 'Explorer' THEN 3
        WHEN behavior_journey_stage_30d = 'Evaluator' THEN 4
        WHEN behavior_journey_stage_30d = 'Builder' THEN 5
        ELSE 1
    END AS current_journey_rank_30d,

    activity_volume_band,
    intent_signal,
    trend_signal,
    recent_activity_trend_ratio
FROM behavior
""")

print('Behavior journey stage distribution')
display(con.execute("""
SELECT
    behavior_journey_stage_30d,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_journey_state_v2
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

print('Combined journey state distribution')
display(con.execute("""
SELECT
    current_journey_state_30d,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_journey_state_v2
GROUP BY 1
ORDER BY developers DESC
LIMIT 30
""").fetchdf())

print('Journey vs dormancy should no longer be one-to-one')
display(con.execute("""
SELECT
    js.behavior_journey_stage_30d,
    d.dormancy_status,
    COUNT(*) AS developers
FROM dev_journey_state_v2 js
LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
GROUP BY 1,2
ORDER BY developers DESC
LIMIT 50
""").fetchdf())

# Guardrails
print('Guardrail: Unactivated should mean no lifetime activity')
display(con.execute("""
SELECT COUNT(*) AS unactivated_with_lifetime_activity
FROM dev_journey_state_v2 j
JOIN dev_features_lifetime_v2 l USING (developer_id)
WHERE j.behavior_journey_stage_30d = 'Unactivated'
  AND l.lifetime_activity_count > 0
""").fetchdf())

Behavior journey stage distribution


,behavior_journey_stage_30d,developers,pct
0,Historically_Active,6945840,74.04
1,Unactivated,1721230,18.35
2,Builder,320215,3.41
3,Learner,269740,2.88
4,Explorer,96094,1.02
5,Evaluator,28389,0.30


Combined journey state distribution


,current_journey_state_30d,developers,pct
0,Dormant_Historically_Active,5116194,54.53
1,Unactivated,1721230,18.35
2,At_Risk_Historically_Active,1498781,15.98
3,Cooling_Historically_Active,330865,3.53
4,Learner,269740,2.88
5,Dormant_Builder,168152,1.79
6,Explorer,96094,1.02
7,At_Risk_Builder,77295,0.82
8,Builder,50311,0.54
9,Cooling_Builder,24457,0.26


Journey vs dormancy should no longer be one-to-one


,behavior_journey_stage_30d,dormancy_status,developers
0,Historically_Active,Dormant,5116194
1,Unactivated,Unactivated,1721230
2,Historically_Active,At_Risk,1498781
3,Historically_Active,Cooling,330865
4,Learner,Active,269740
5,Builder,Dormant,168152
6,Explorer,Active,96094
7,Builder,At_Risk,77295
8,Builder,Active,50311
9,Builder,Cooling,24457


Guardrail: Unactivated should mean no lifetime activity


,unactivated_with_lifetime_activity
0,0


## 13. Final profile: join contact metadata last

Update: Final profile now includes `final_lifecycle_status`, combining `user_type`, max stage reached, and dormancy timing.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dev_profile_final_v4 AS
SELECT
    u.developer_id,

    -- Persona
    p.persona,
    p.persona_confidence,
    p.persona_confidence_tier,
    p.persona_entropy,
    p.mixed_persona_flag,
    p.cuda_share,
    p.genai_share,
    p.robotics_share,
    p.simulation_share,
    p.learning_community_share,

    -- Developer effort segmentation
    e.developer_effort_score,
    e.developer_effort_level,
    e.developer_effort_rank,
    e.recency_weight AS effort_recency_weight,

    -- Journey and dormancy
    js.behavior_journey_stage_30d,
    js.behavior_journey_rank_30d,
    js.current_journey_state_30d,
    js.current_journey_rank_30d,
    d.is_activated,
    d.lifetime_meaningful_weeks,
    d.last_meaningful_week_start,
    d.days_since_last_meaningful_week,
    d.dormancy_status,
    d.dormant_flag,
    d.at_risk_flag,
    CASE
        -- Signed-up contacts with zero lifetime activity are not active users.
        -- Keep this first so they do not fall through to Active_None.
        WHEN COALESCE(lf.lifetime_activity_count, 0) = 0 THEN 'Unactivated'
        WHEN lf.user_type = 'tourist' THEN 'Tourist'
        WHEN lf.user_type = 'free_email_user' THEN 'FreeEmail'
        WHEN COALESCE(d.days_since_last_activity, d.days_since_last_meaningful_week) >= 365 THEN 'Dormant_' || lf.max_stage_reached
        WHEN COALESCE(d.days_since_last_activity, d.days_since_last_meaningful_week) >= 180 THEN 'AtRisk_' || lf.max_stage_reached
        ELSE 'Active_' || lf.max_stage_reached
    END AS final_lifecycle_status,

    -- Recency / incremental windows
    r.* EXCLUDE (developer_id),

    -- Lifetime features
    lf.* EXCLUDE (developer_id, cuda_score, genai_score, robotics_score, simulation_score, learning_community_score, other_persona_score),

    -- Contact enrichment, intentionally last
    c.created_date AS contact_created_date,
    c.first_activity_date AS contact_first_activity_date,
    c.last_activity_date AS contact_last_activity_date,
    c.account_id,
    c.account_type,
    c.country,
    c.region,
    c.industry_segment_vertical,
    c.program_application_source,
    c.organization_english_name,
    c.normalized_account_name,
    c.wwfo_category,
    c.wwfo_target_list,
    CASE WHEN c.developer_id IS NULL THEN 1 ELSE 0 END AS missing_contact_metadata_flag
FROM developer_universe_v2 u
LEFT JOIN dev_persona_v2 p USING (developer_id)
LEFT JOIN dev_effort_level_v2 e USING (developer_id)
LEFT JOIN dev_journey_state_v2 js USING (developer_id)
LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
LEFT JOIN dev_recency_features_v2 r USING (developer_id)
LEFT JOIN dev_features_lifetime_v2 lf USING (developer_id)
LEFT JOIN contact_one_row_v2 c USING (developer_id)
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    SUM(CASE WHEN persona IS NULL THEN 1 ELSE 0 END) AS missing_persona,
    SUM(CASE WHEN current_journey_state_30d IS NULL THEN 1 ELSE 0 END) AS missing_journey_state,
    SUM(missing_contact_metadata_flag) AS missing_contact_metadata
FROM dev_profile_final_v4
""").fetchdf())

display(con.execute("""
SELECT persona, current_journey_state_30d, dormancy_status, final_lifecycle_status, COUNT(*) AS developers
FROM dev_profile_final_v4
GROUP BY 1,2,3,4
ORDER BY developers DESC
LIMIT 40
""").fetchdf())


100% ▕██████████████████████████████████████▏ (00:00:19.05 elapsed)     


,rows,developers,missing_persona,missing_journey_state,missing_contact_metadata
0,9381508,9381508,0.0,0.0,18.0


,persona,current_journey_state_30d,dormancy_status,final_lifecycle_status,developers
0,CUDA,Dormant_Historically_Active,Dormant,Tourist,1504859
1,Unknown,Unactivated,Unactivated,Unactivated,1293261
2,GenAI,Dormant_Historically_Active,Dormant,Tourist,850449
3,CUDA,Dormant_Historically_Active,Dormant,Dormant_Build,738979
4,GenAI,At_Risk_Historically_Active,At_Risk,Tourist,413976
5,Unknown,Dormant_Historically_Active,Dormant,Tourist,361219
6,CUDA,At_Risk_Historically_Active,At_Risk,Tourist,352447
7,Simulation,Dormant_Historically_Active,Dormant,Tourist,336864
8,GenAI,Unactivated,Unactivated,Unactivated,283810
9,Learning_Community,Dormant_Historically_Active,Dormant,Tourist,236488


## 15. Basic EDA of final developer profile

These cells give the lightweight final-data EDA needed before modeling: final row coverage, effort-level distribution, lifecycle distribution, segment quality by geography/account type, activity mix by effort level, and extreme outlier checks.


In [ ]:

print('Final profile shape and coverage')
display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    SUM(CASE WHEN missing_contact_metadata_flag = 1 THEN 1 ELSE 0 END) AS missing_contact_metadata,
    ROUND(100.0 * SUM(CASE WHEN missing_contact_metadata_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_contact_metadata_pct,
    MIN(lifetime_first_activity_date) AS min_lifetime_first_activity_date,
    MAX(lifetime_last_activity_date) AS max_lifetime_last_activity_date
FROM dev_profile_final_v4
""").fetchdf())

print('Effort level summary')
display(con.execute("""
SELECT
    developer_effort_level,
    developer_effort_rank,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_developers,
    ROUND(AVG(developer_effort_score), 3) AS avg_effort_score,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_lifetime_activity_count,
    ROUND(AVG(lifetime_activity_score_sum), 2) AS avg_lifetime_activity_score_sum,
    ROUND(AVG(lifetime_unique_activity_types), 2) AS avg_lifetime_activity_types,
    ROUND(AVG(lifetime_unique_modalities), 2) AS avg_lifetime_modalities
FROM dev_profile_final_v4
GROUP BY 1,2
ORDER BY developer_effort_rank DESC
""").fetchdf())

print('Activity mix by effort level')
display(con.execute("""
SELECT
    developer_effort_level,
    COUNT(*) AS developers,
    ROUND(AVG(lifetime_dli_training_count), 2) AS avg_dli_training,
    ROUND(AVG(lifetime_webinar_count), 2) AS avg_webinars,
    ROUND(AVG(lifetime_devzone_download_count), 2) AS avg_devzone_downloads,
    ROUND(AVG(lifetime_ngc_download_count), 2) AS avg_ngc_downloads,
    ROUND(AVG(lifetime_api_count), 2) AS avg_api,
    ROUND(AVG(lifetime_forum_count), 2) AS avg_forum,
    ROUND(AVG(lifetime_bug_count), 2) AS avg_bugs,
    ROUND(AVG(lifetime_hackathon_count), 2) AS avg_hackathons
FROM dev_profile_final_v4
GROUP BY 1, developer_effort_rank
ORDER BY developer_effort_rank DESC
""").fetchdf())

print('Lifecycle and effort cross-tab')
display(con.execute("""
SELECT
    developer_effort_level,
    final_lifecycle_status,
    COUNT(*) AS developers
FROM dev_profile_final_v4
GROUP BY 1,2, developer_effort_rank
ORDER BY developer_effort_rank DESC, developers DESC
LIMIT 50
""").fetchdf())

print('Top outliers by raw activity count')
display(con.execute("""
SELECT
    developer_id,
    developer_effort_level,
    developer_effort_score,
    lifetime_activity_count,
    lifetime_activity_score_sum,
    lifetime_unique_activity_types,
    lifetime_devzone_download_count,
    lifetime_ngc_download_count,
    lifetime_api_count,
    final_lifecycle_status,
    country,
    account_type,
    normalized_account_name
FROM dev_profile_final_v4
ORDER BY lifetime_activity_count DESC
LIMIT 25
""").fetchdf())


Final profile shape and coverage


,rows,developers,missing_contact_metadata,missing_contact_metadata_pct,min_lifetime_first_activity_date,max_lifetime_last_activity_date
0,9381508,9381508,18.0,0.0,2020-01-01,2026-03-12


Effort level summary


,developer_effort_level,developer_effort_rank,developers,pct_developers,avg_effort_score,avg_lifetime_activity_count,avg_lifetime_activity_score_sum,avg_lifetime_activity_types,avg_lifetime_modalities
0,very high effort,4,766031,8.17,1.226,57.20,180.79,2.56,1.98
1,high effort,3,1192351,12.71,0.550,9.57,47.60,2.34,1.90
2,medium effort,2,1950701,20.79,0.389,3.69,14.85,2.29,1.89
3,low effort,1,3751195,39.98,0.233,1.85,3.81,1.53,1.47
4,no activity,0,1721230,18.35,0.000,0.00,0.00,0.00,0.00


Activity mix by effort level


,developer_effort_level,developers,avg_dli_training,avg_webinars,avg_devzone_downloads,avg_ngc_downloads,avg_api,avg_forum,avg_bugs,avg_hackathons
0,very high effort,766031,0.74,0.13,35.15,9.68,7.44,0.85,0.11,0.0
1,high effort,1192351,0.70,0.07,5.92,0.18,0.16,0.07,0.02,0.0
2,medium effort,1950701,0.27,0.02,1.75,0.02,0.03,0.01,0.00,0.0
3,low effort,3751195,0.00,0.02,0.62,0.01,0.14,0.01,0.00,0.0
4,no activity,1721230,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0


Lifecycle and effort cross-tab


,developer_effort_level,final_lifecycle_status,developers
0,very high effort,Tourist,231916
1,very high effort,Active_Build,185789
2,very high effort,Active_Discover,116238
3,very high effort,Active_Learn,74885
4,very high effort,Dormant_Build,67500
5,very high effort,FreeEmail,25388
6,very high effort,AtRisk_Build,21616
7,very high effort,Dormant_Champion,14316
8,very high effort,Active_Champion,14054
9,very high effort,Active_Evaluate,4578


Top outliers by raw activity count


,developer_id,developer_effort_level,developer_effort_score,lifetime_activity_count,lifetime_activity_score_sum,lifetime_unique_activity_types,lifetime_devzone_download_count,lifetime_ngc_download_count,lifetime_api_count,final_lifecycle_status,country,account_type,normalized_account_name
0,5903679,very high effort,4.330158,2027375,10015777.0,2,0.0,2027374.0,0.0,Active_Build,UNITED STATES,enterprise,NVIDIA
1,5886356,very high effort,4.343515,789400,3946993.0,4,1.0,789395.0,0.0,Active_Build,SINGAPORE,enterprise,Straitdeer Pte Ltd
2,3931685,very high effort,3.510924,187095,935275.0,1,0.0,187095.0,0.0,Active_Build,PORTUGAL,startup,Onfido
3,1303597,very high effort,3.837162,151947,450792.0,3,151937.0,0.0,0.0,Active_Build,INDIA,enterprise,NVIDIA
4,2028858,very high effort,3.702349,142980,714580.0,1,0.0,142980.0,0.0,Active_Build,UNITED STATES,unknown,University of Missouri
5,6309430,very high effort,4.002649,121950,235235.0,5,656.0,121180.0,102.0,Active_Build,UNITED STATES,enterprise,NVIDIA
6,4634152,very high effort,1.520438,107089,180756.0,1,0.0,107089.0,0.0,Active_Build,UNITED STATES,unknown,Unclassified - Invalid
7,6125204,very high effort,3.743236,68992,344961.0,9,31.0,68888.0,0.0,Active_Build,THAILAND,unknown,Not Normalized
8,4833930,very high effort,0.785561,55061,275414.0,2,0.0,55060.0,0.0,Dormant_Build,UNITED STATES,enterprise,NVIDIA
9,225774,very high effort,0.760219,53677,268041.0,1,0.0,53677.0,0.0,Dormant_Build,UNITED STATES,enterprise,Paige ai


## 14. Feature validation checklist

This validates final profile shape, feature ranges, journey/dormancy separation, and dictionary determinism before modeling.


In [ ]:
validation = con.execute("""
WITH dict_check AS (
    SELECT COUNT(*) AS inconsistent_activity_labels
    FROM (
        SELECT activity
        FROM activity_labeled_v2
        GROUP BY activity
        HAVING COUNT(DISTINCT journey_signal) > 1
            OR COUNT(DISTINCT effort_level) > 1
            OR COUNT(DISTINCT modality) > 1
    ) x
),
profile_check AS (
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT developer_id) AS distinct_developers,
        SUM(CASE WHEN activity_count_0_30d < 0 OR lifetime_activity_count < 0 THEN 1 ELSE 0 END) AS negative_count_violations,
        SUM(CASE WHEN persona_confidence < 0 OR persona_confidence > 1 THEN 1 ELSE 0 END) AS persona_confidence_violations,
        SUM(CASE WHEN persona_entropy < -0.000001 OR persona_entropy > 1.000001 THEN 1 ELSE 0 END) AS persona_entropy_violations,
        SUM(CASE WHEN has_activity_0_30d = 0 AND activity_count_0_30d <> 0 THEN 1 ELSE 0 END) AS zero_activity_flag_violations,
        SUM(CASE WHEN recent_build_flag = 1 AND build_count_0_30d = 0 THEN 1 ELSE 0 END) AS recent_build_flag_violations,
        SUM(CASE WHEN missing_contact_metadata_flag NOT IN (0,1) THEN 1 ELSE 0 END) AS contact_flag_violations,
        SUM(CASE WHEN developer_effort_score < 0 THEN 1 ELSE 0 END) AS developer_effort_score_violations,
        SUM(CASE WHEN developer_effort_rank NOT BETWEEN 0 AND 4 THEN 1 ELSE 0 END) AS developer_effort_rank_violations,
        SUM(CASE WHEN behavior_journey_stage_30d = 'Unactivated' AND lifetime_activity_count > 0 THEN 1 ELSE 0 END) AS unactivated_lifetime_activity_violations,
        SUM(CASE WHEN final_lifecycle_status = 'Unactivated' AND lifetime_activity_count > 0 THEN 1 ELSE 0 END) AS final_unactivated_lifetime_activity_violations,
        SUM(CASE WHEN lifetime_activity_count = 0 AND final_lifecycle_status <> 'Unactivated' THEN 1 ELSE 0 END) AS zero_lifetime_not_unactivated_violations
    FROM dev_profile_final_v4
)
SELECT *
FROM profile_check
CROSS JOIN dict_check
""").fetchdf()

display(validation)

if validation.loc[0, "rows"] != validation.loc[0, "distinct_developers"]:
    raise ValueError("Final table is not one row per developer")

violation_cols = [c for c in validation.columns if c.endswith("violations") or c == "inconsistent_activity_labels"]
violations = validation.loc[0, violation_cols].sum()
if violations > 0:
    print("WARNING: validation violations found. Inspect the table above before modeling.")
else:
    print("All feature validation checks passed.")


,rows,distinct_developers,negative_count_violations,persona_confidence_violations,persona_entropy_violations,zero_activity_flag_violations,recent_build_flag_violations,contact_flag_violations,developer_effort_score_violations,developer_effort_rank_violations,unactivated_lifetime_activity_violations,final_unactivated_lifetime_activity_violations,zero_lifetime_not_unactivated_violations,inconsistent_activity_labels
0,9381508,9381508,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


## 15. Table inventory

In [ ]:
final_tables = [
    "activity_base_v2",
    "contact_one_row_v2",
    "activity_labeled_v2",
    "developer_universe_v2",
    "dev_features_0_30d_v2",
    "dev_features_30_90d_v2",
    "dev_features_90_180d_v2",
    "dev_recency_features_v2",
    "dev_features_lifetime_v2",
    "dev_effort_level_v2",
    "dev_contact_persona_v2",
    "dev_persona_v2",
    "dev_journey_state_v2",
    "dev_weekly_features_v2",
    "dev_meaningful_week_v2",
    "dev_activation_v2",
    "dev_dormancy_status_v2",
    "dev_profile_final_v4",
]

inventory = []
for t in final_tables:
    exists = con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?", [t]).fetchone()[0] > 0
    rows = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] if exists else None
    inventory.append({"table": t, "rows": rows})

display(pd.DataFrame(inventory))


,table,rows
0,activity_base_v2,69347501
1,contact_one_row_v2,9381490
2,activity_labeled_v2,69347501
3,developer_universe_v2,9381508
4,dev_features_0_30d_v2,9381508
5,dev_features_30_90d_v2,9381508
6,dev_features_90_180d_v2,9381508
7,dev_recency_features_v2,9381508
8,dev_features_lifetime_v2,9381508
9,dev_effort_level_v2,9381508


In [ ]:
# Close when finished.
con.close()